**Analisis Segmentasi dan Prediksi Kualitas Pembangunan Daerah Berbasis Data Sosial-Ekonomi Indonesia**

## Alur Pengerjaan

1. Anggota 1: Data preparation dan development score (Radiv)
2. Anggota 2: EDA dan visualisasi (Imanuel)
3. Anggota 3: K-Means clustering (Zhafir)
4. Anggota 4: Random Forest, evaluasi, dan hasil akhir (Naufal)

# Bagian Anggota 1: Data Preparation

## 1. Import library

In [ ]:
import pandas as pd
import numpy as np

## 2. Load dataset

In [ ]:
df = pd.read_csv("2021socio_economic_indonesia.csv")
df.head()

## 3. Cek struktur data

In [ ]:
df.info()
df.describe()
df.shape

## 4. Cek missing value

In [ ]:
df.isnull().sum()

## 5. Cek duplikasi

In [ ]:
df.duplicated().sum()

## 6. Cleaning data

In [ ]:
df = df.drop_duplicates()
df = df.dropna()
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.replace(",", ".", regex=False)

## 7. Pilih fitur utama

In [ ]:
fitur_utama = [
    "poorpeople_percentage",
    "avg_schooltime",
    "life_exp",
    "exp_percap"
]

## 8. Validasi tipe data dan pastikan fitur numerik

In [ ]:
for col in fitur_utama:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=fitur_utama)
print("Tipe data fitur utama:")
print(df[fitur_utama].dtypes)

# Bagian Anggota 2: EDA dan Visualisasi

## 1. Import library

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Statistik deskriptif

In [ ]:
df.describe()

## 3. Distribusi kemiskinan

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["poorpeople_percentage"], kde=True)
plt.title("Distribusi Persentase Penduduk Miskin")
plt.xlabel("Persentase Penduduk Miskin")
plt.ylabel("Jumlah Daerah")
plt.show()

## 4. Top 10 daerah dengan kemiskinan tertinggi

In [ ]:
top_miskin = df.sort_values(
    "poorpeople_percentage",
    ascending=False
).head(10)

plt.figure(figsize=(10,6))
sns.barplot(
    data=top_miskin,
    x="poorpeople_percentage",
    y="cities_reg"
)
plt.title("Top 10 Daerah dengan Kemiskinan Tertinggi")
plt.show()

## 5. Scatter plot pendidikan vs pengeluaran

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df,
    x="avg_schooltime",
    y="exp_percap"
)
plt.title("Pendidikan vs Pengeluaran Per Kapita")
plt.xlabel("Rata-rata Lama Sekolah")
plt.ylabel("Pengeluaran Per Kapita")
plt.show()

## 6. Scatter plot kemiskinan vs pengeluaran

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df,
    x="poorpeople_percentage",
    y="exp_percap"
)
plt.title("Kemiskinan vs Pengeluaran Per Kapita")
plt.xlabel("Persentase Penduduk Miskin")
plt.ylabel("Pengeluaran Per Kapita")
plt.show()

## 7. Heatmap korelasi

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(
    df.select_dtypes(include="number").corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Korelasi Antar Variabel")
plt.show()

# Bagian Anggota 3: K-Means Clustering

## 1. Import library

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 2. Pilih fitur clustering

In [ ]:
features_cluster = [
    "poorpeople_percentage",
    "reg_gdp",
    "life_exp",
    "avg_schooltime",
    "exp_percap"
]

## 3. Standardisasi data

In [ ]:
X_cluster = df[features_cluster]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

## 4. Elbow Method

In [ ]:
inertia = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(2, 8), inertia, marker="o")
plt.title("Elbow Method untuk Menentukan Jumlah Cluster")
plt.xlabel("Jumlah Cluster")
plt.ylabel("Inertia")
plt.show()

## 5. Silhouette Score untuk berbagai nilai k

In [ ]:
silhouette_scores = []

for k in range(2, 8):
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = kmeans_temp.fit_predict(X_scaled)
    score_temp = silhouette_score(X_scaled, labels_temp)
    silhouette_scores.append(score_temp)
    print(f"k={k} -> Silhouette Score: {score_temp:.4f}")

plt.figure(figsize=(8,5))
plt.plot(range(2, 8), silhouette_scores, marker="o", color="orange")
plt.title("Silhouette Score untuk Setiap Nilai k")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Silhouette Score")
plt.xticks(range(2, 8))
plt.show()

best_k = range(2, 8)[silhouette_scores.index(max(silhouette_scores))]
print(f"\nk={best_k} dipilih karena menghasilkan Silhouette Score tertinggi ({max(silhouette_scores):.4f}), menunjukkan cluster yang paling kohesif dan terpisah.")
print("Nilai ini konsisten dengan k=3 yang digunakan pada proyek ini.")

## 6. Jalankan K-Means dengan k=3

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

score = silhouette_score(X_scaled, df["cluster"])
print("Silhouette Score (k=3):", round(score, 4))

## 7. Lihat jumlah daerah per cluster

In [ ]:
df["cluster"].value_counts()

## 8. Visualisasi jumlah daerah per cluster

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="cluster")
plt.title("Jumlah Daerah per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Jumlah Daerah")
plt.show()

## 9. Ringkasan karakteristik cluster

In [ ]:
cluster_summary = df.groupby("cluster")[features_cluster].mean()
print("Rata-rata indikator per cluster:")
cluster_summary

## 10. Interpretasi dan pelabelan cluster

In [ ]:
# Tentukan label berdasarkan composite score dari semua indikator cluster
# Indikator positif (semakin tinggi = semakin baik): reg_gdp, life_exp, avg_schooltime, exp_percap
# Indikator negatif (semakin rendah = semakin baik): poorpeople_percentage

from sklearn.preprocessing import MinMaxScaler as MMS

cluster_means = df.groupby("cluster")[features_cluster].mean()

# Normalisasi rata-rata per cluster untuk composite scoring
scaler_label = MMS()
cluster_means_norm = pd.DataFrame(
    scaler_label.fit_transform(cluster_means),
    columns=features_cluster,
    index=cluster_means.index
)

# Balik kemiskinan: rendah = lebih baik
cluster_means_norm["poorpeople_percentage"] = 1 - cluster_means_norm["poorpeople_percentage"]

# Composite score = rata-rata semua indikator yang sudah dinormalisasi
cluster_means_norm["composite_score"] = cluster_means_norm.mean(axis=1)

print("Composite score per cluster (berdasarkan semua indikator):")
print(cluster_means_norm[["composite_score"]].round(4))
print()

# Urutkan cluster berdasarkan composite score
sorted_by_composite = cluster_means_norm["composite_score"].sort_values().index.tolist()

label_map = {
    sorted_by_composite[0]: "Pembangunan Rendah",
    sorted_by_composite[1]: "Pembangunan Sedang",
    sorted_by_composite[2]: "Pembangunan Tinggi"
}

df["label_cluster"] = df["cluster"].map(label_map)

print("Pemetaan cluster ke label:")
for k, v in label_map.items():
    print(f"  Cluster {k} -> {v} (composite score: {cluster_means_norm.loc[k, 'composite_score']:.4f})")

print()
print("Narasi pelabelan cluster:")
print(cluster_means.round(2).to_string())
print()
print("Label ditentukan berdasarkan composite score dari 5 indikator:")
print("  - poorpeople_percentage (dibalik: rendah = lebih baik)")
print("  - reg_gdp, life_exp, avg_schooltime, exp_percap (tinggi = lebih baik)")
print("Cluster dengan composite score tertinggi dilabeli 'Pembangunan Tinggi',")
print("cluster dengan composite score terendah dilabeli 'Pembangunan Rendah'.")


## 11. Tabel ringkasan cluster dengan label

In [ ]:
cluster_labeled_summary = df.groupby("label_cluster")[features_cluster].mean().round(2)
print("Rata-rata indikator per kategori pembangunan:")
cluster_labeled_summary

## 12. Visualisasi scatterplot cluster dengan centroid

In [ ]:
# Hitung centroid dalam ruang asli (inverse transform)
centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids_original, columns=features_cluster)
centroids_df["cluster"] = range(3)
centroids_df["label_cluster"] = centroids_df["cluster"].map(label_map)

# Scatterplot cluster: avg_schooltime vs exp_percap
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="avg_schooltime",
    y="exp_percap",
    hue="label_cluster",
    palette="Set2",
    alpha=0.7
)

# Tambahkan centroid
plt.scatter(
    centroids_df["avg_schooltime"],
    centroids_df["exp_percap"],
    marker="X",
    s=200,
    c="black",
    zorder=5,
    label="Centroid"
)

plt.title("Cluster K-Means: Pendidikan vs Pengeluaran Per Kapita")
plt.xlabel("Rata-rata Lama Sekolah")
plt.ylabel("Pengeluaran Per Kapita")
plt.legend(title="Kategori")
plt.show()

In [ ]:
# Scatterplot cluster: poorpeople_percentage vs exp_percap
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="poorpeople_percentage",
    y="exp_percap",
    hue="label_cluster",
    palette="Set2",
    alpha=0.7
)

# Tambahkan centroid
plt.scatter(
    centroids_df["poorpeople_percentage"],
    centroids_df["exp_percap"],
    marker="X",
    s=200,
    c="black",
    zorder=5,
    label="Centroid"
)

plt.title("Cluster K-Means: Kemiskinan vs Pengeluaran Per Kapita")
plt.xlabel("Persentase Penduduk Miskin")
plt.ylabel("Pengeluaran Per Kapita")
plt.legend(title="Kategori")
plt.show()

# Bagian Anggota 4: Random Forest dan Evaluasi

## 1. Import library

In [ ]:
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc
)
from itertools import cycle

## 2. Buat target klasifikasi untuk Random Forest

In [ ]:
# Buat skor pembangunan menggunakan MinMaxScaler pada fitur utama
scaler_mm = MinMaxScaler()
scaled_rf = pd.DataFrame(
    scaler_mm.fit_transform(df[fitur_utama]),
    columns=fitur_utama,
    index=df.index
)

# Kemiskinan dikurangi karena semakin rendah = semakin baik
scaled_rf["poorpeople_percentage"] = 1 - scaled_rf["poorpeople_percentage"]
df["skor_pembangunan"] = scaled_rf.mean(axis=1)

# Buat target kategori pembangunan menggunakan quantile cut
df["target_kategori_pembangunan"] = pd.qcut(
    df["skor_pembangunan"],
    q=3,
    labels=["Rendah", "Sedang", "Tinggi"]
)

print("Distribusi target_kategori_pembangunan:")
print(df["target_kategori_pembangunan"].value_counts())
print()
print("[CATATAN METODOLOGI]")
print("Target 'target_kategori_pembangunan' dibentuk dari fitur yang sama")
print("(poorpeople_percentage, avg_schooltime, life_exp, exp_percap) yang")
print("juga digunakan sebagai fitur input model Random Forest.")
print("Ini berpotensi menyebabkan target leakage, sehingga akurasi model")
print("mungkin terlihat sangat tinggi karena target dan fitur bersumber dari data yang sama.")
print()
print("Rekomendasi (opsional untuk pengembangan selanjutnya):")
print("  - Gunakan fitur lain yang tidak masuk ke pembentukan skor sebagai prediktor")
print("  - Atau gunakan label cluster K-Means sebagai target alternatif")
print()
print("Untuk konteks tugas kuliah Data Mining, pendekatan ini MASIH LAYAK karena:")
print("  1. Tujuan utama adalah demonstrasi pipeline supervised learning")
print("  2. Labeling berbasis composite score adalah pendekatan umum dalam proyek akademik")
print("  3. Model tetap menghasilkan output yang dapat diinterpretasikan")


## 3. Tentukan fitur dan target

In [ ]:
features_model = [
    "poorpeople_percentage",
    "avg_schooltime",
    "life_exp",
    "exp_percap"
]

X = df[features_model]
y = df["target_kategori_pembangunan"]

## 4. Split data train-test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 5. Buat dan latih model Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
model.fit(X_train, y_train)

## 6. Prediksi dan evaluasi

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print()
print(classification_report(y_test, y_pred))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=model.classes_,
    yticklabels=model.classes_
)
plt.title("Confusion Matrix")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.show()

## 8. Feature importance

In [ ]:
importance = pd.DataFrame({
    "Fitur": features_model,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=importance, x="Importance", y="Fitur")
plt.title("Feature Importance Random Forest")
plt.xlabel("Importance")
plt.ylabel("Fitur")
plt.show()

importance

## 9. ROC AUC - One-vs-Rest Multiclass

In [ ]:
# Binarize label untuk One-vs-Rest
classes = model.classes_
y_test_bin = label_binarize(y_test, classes=classes)
y_prob = model.predict_proba(X_test)

# Hitung ROC dan AUC per kelas
fpr = {}
tpr = {}
roc_auc = {}

for i, cls in enumerate(classes):
    fpr[cls], tpr[cls], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc[cls] = auc(fpr[cls], tpr[cls])

# Plot ROC curve tiap kelas
plt.figure(figsize=(9, 6))
colors = cycle(["blue", "orange", "green"])

for cls, color in zip(classes, colors):
    plt.plot(
        fpr[cls],
        tpr[cls],
        color=color,
        lw=2,
        label=f"Kelas '{cls}' (AUC = {roc_auc[cls]:.2f})"
    )

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve per Kelas (One-vs-Rest)")
plt.legend(loc="lower right")
plt.show()

print("AUC per kelas:")
for cls in classes:
    print(f"  {cls}: {roc_auc[cls]:.4f}")

# Macro-average AUC
macro_auc = np.mean(list(roc_auc.values()))
print(f"\nROC AUC Macro-Average: {macro_auc:.4f}")

## 10. Simpan hasil akhir

In [ ]:
df.to_csv("socio_economic_final.csv", index=False)
print("Dataset final berhasil disimpan ke socio_economic_final.csv")